# 00 · Basics
Learning notebook for the Semantic AI Gateway & Cache. No product code here.

## 1. Embeddings
An embedding turns a sentence into a list of numbers, arranged so that sentences with similar meaning end up pointing in similar directions. Here each sentence becomes one such list.

In [1]:
import numpy as np
from sentence_transformers import SentenceTransformer

# CPU on purpose: the design runs embedding on CPU, so the timings later match that.
model = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cpu")

W1125 12:41:20.091000 18255 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [2]:
sentences = [
    "How do I reset my password?",
    "Password reset steps?",
    "I forgot my password, what do I do?",
    "How do I reset my username?",
    "How do I cancel my order?",
    "What's the weather today?",
]

# normalize_embeddings=True scales every vector to length 1.
embeddings = model.encode(sentences, normalize_embeddings=True)

print("shape:", embeddings.shape)  # (number of sentences, numbers per sentence)

print("L2 norm of first vector:", np.linalg.norm(embeddings[0]))

shape: (6, 384)
L2 norm of first vector: 1.0


In [3]:
import numpy as np
np.set_printoptions(precision=3, suppress=True, linewidth=150)

# 1. The full table: 6 rows (sentences) x 384 columns (numbers)
print("shape:", embeddings.shape)

# 2. The first 8 numbers of each sentence, side by side
for sentence, vector in zip(sentences, embeddings):
    print(f"{sentence:40s} {vector[:8]} ... ({len(vector)} numbers)")

# 3. All 384 numbers of ONE sentence
print("\nAll 384 numbers for:", sentences[0])
print(embeddings[0])

shape: (6, 384)
How do I reset my password?              [-0.024 -0.03   0.028 -0.059 -0.061 -0.001  0.047 -0.045] ... (384 numbers)
Password reset steps?                    [-0.049 -0.044  0.024 -0.067 -0.092  0.029  0.003 -0.045] ... (384 numbers)
I forgot my password, what do I do?      [-0.021 -0.014  0.025 -0.08  -0.066 -0.022  0.069 -0.016] ... (384 numbers)
How do I reset my username?              [ 0.013  0.004  0.056 -0.056 -0.028  0.018  0.066 -0.039] ... (384 numbers)
How do I cancel my order?                [ 0.003 -0.006  0.071  0.002 -0.022 -0.024  0.042 -0.014] ... (384 numbers)
What's the weather today?                [-0.032 -0.07  -0.008  0.025  0.076  0.017  0.029 -0.018] ... (384 numbers)

All 384 numbers for: How do I reset my password?
[-0.024 -0.03   0.028 -0.059 -0.061 -0.001  0.047 -0.045  0.021 -0.021 -0.005  0.007  0.05  -0.037  0.106  0.054 -0.037  0.042 -0.041  0.045 -0.018
  0.003 -0.001 -0.021 -0.063  0.01  -0.055 -0.013 -0.083 -0.082 -0.036 -0.009 -0.005

## 2. Cosine similarity
It measures how closely two vectors point in the same direction. Higher means closer.

$$\text{sim}(A,B)=\frac{A\cdot B}{\lVert A\rVert\,\lVert B\rVert}$$

Our vectors are normalized, so $\lVert A\rVert=\lVert B\rVert=1$, the bottom of the fraction is 1, and the similarity is just the dot product $A\cdot B$.

In [4]:
# (6 x 384) @ (384 x 6) gives a 6 x 6 table: every sentence against every sentence.
similarity = embeddings @ embeddings.T

for i, sentence in enumerate(sentences):
    print(f"[{i}] {sentence}")
print()
print(" " * 40 + "".join(f"    [{i}]" for i in range(len(sentences))))
for i, sentence in enumerate(sentences):
    row = "".join(f"{score:7.3f}" for score in similarity[i])
    print(f"[{i}] {sentence:36}{row}")

[0] How do I reset my password?
[1] Password reset steps?
[2] I forgot my password, what do I do?
[3] How do I reset my username?
[4] How do I cancel my order?
[5] What's the weather today?

                                            [0]    [1]    [2]    [3]    [4]    [5]
[0] How do I reset my password?           1.000  0.920  0.913  0.863  0.551  0.358
[1] Password reset steps?                 0.920  1.000  0.834  0.781  0.541  0.319
[2] I forgot my password, what do I do?   0.913  0.834  1.000  0.770  0.521  0.342
[3] How do I reset my username?           0.863  0.781  0.770  1.000  0.580  0.377
[4] How do I cancel my order?             0.551  0.541  0.521  0.580  1.000  0.415
[5] What's the weather today?             0.358  0.319  0.342  0.377  0.415  1.000


**What to notice**
- The real paraphrases here score well below 0.98: the three password-reset sentences score 0.834 to 0.920 against each other. A 0.98 cut-off would miss all of them.
- "Reset password" vs "reset username" scores 0.863 although the meaning is different. That is higher than the 0.834 between two real paraphrases ([1] and [2]). So on this example, no single threshold can accept every paraphrase pair and also reject this wrong one.
- This is why the project needs a tuned threshold **plus a verifier**. Six sentences only illustrate the problem; the threshold itself will be chosen from a much larger labelled set.

## 3. Brute-force nearest neighbour
To find a cached answer, compare the new question with every stored one and keep the closest.

In [5]:
def nearest(query, stored):
    query_vector = model.encode(query, normalize_embeddings=True)
    stored_vectors = model.encode(stored, normalize_embeddings=True)  # a real cache saves these once
    scores = stored_vectors @ query_vector  # one similarity per stored sentence
    best = np.argmax(scores)  # position of the highest score
    return stored[best], float(scores[best])

In [6]:
for query in ["how can i change my password", "where is my package"]:
    match, score = nearest(query, sentences)
    print(f"query:   {query}")
    print(f"nearest: {match}  (similarity {score:.3f})")
    print()

query:   how can i change my password
nearest: How do I reset my password?  (similarity 0.895)

query:   where is my package
nearest: How do I cancel my order?  (similarity 0.638)



Nearest does not mean same meaning: this function always returns *something*, so the score still has to pass a check. Real systems use an index (HNSW in Redis) because comparing against every vector gets slower as the cache grows, while HNSW visits only a small part of them and accepts that it may occasionally miss the true nearest one.

## 4. The OpenAI chat format (via Ollama)
Ollama accepts the same request and response shape as OpenAI's API. Our gateway will sit in the middle and speak this shape on both sides.

In [8]:
import json
import httpx

URL = "http://localhost:11434/v1/chat/completions"
request_body = {
    "model": "qwen2.5:1.5b",
    "temperature": 0,
    "messages": [{"role": "user", "content": "How do I reset my password?"}],
    
}

# httpx gives up after 5 seconds by default, which is too short for a generation.
response = httpx.post(URL, json=request_body, timeout=120)
response.raise_for_status()  # stop here on an HTTP error
data = response.json()

print(json.dumps(data, indent=2))

{
  "id": "chatcmpl-821",
  "object": "chat.completion",
  "created": 1795590745,
  "model": "qwen2.5:1.5b",
  "system_fingerprint": "fp_ollama",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "To reset your password, you can follow these steps:\n\n1. Open your email account's website or app.\n2. Look for a \"Forgot Password\" or \"Reset Password\" link or button.\n3. Click on it to start the password reset process.\n4. You will be prompted to enter your email address associated with your account.\n5. You will receive a password reset link or code in your email.\n6. Follow the instructions provided in the email to reset your password. This may involve clicking on a link, entering a code, or answering a security question.\n7. Once you have reset your password, you can log in to your account using the new password.\n\nIf you are unable to find a \"Forgot Password\" link or button, you may need to contact your email provider's cus

In [9]:
choice = data["choices"][0]

print("content:")
print(choice["message"]["content"])
print()
print("finish_reason:", choice["finish_reason"])
print("usage:", data["usage"])

content:
To reset your password, you can follow these steps:

1. Open your email account's website or app.
2. Look for a "Forgot Password" or "Reset Password" link or button.
3. Click on it to start the password reset process.
4. You will be prompted to enter your email address associated with your account.
5. You will receive a password reset link or code in your email.
6. Follow the instructions provided in the email to reset your password. This may involve clicking on a link, entering a code, or answering a security question.
7. Once you have reset your password, you can log in to your account using the new password.

If you are unable to find a "Forgot Password" link or button, you may need to contact your email provider's customer support for assistance.

finish_reason: stop
usage: {'prompt_tokens': 36, 'prompt_tokens_details': {'cached_tokens': 0}, 'completion_tokens': 166, 'total_tokens': 202}


- `choices[0].message.content`: the answer text.
- `choices[0].finish_reason`: why the model stopped writing.
- `usage`: token counts: `prompt_tokens` (input), `completion_tokens` (generated), `total_tokens` (both added).

`finish_reason` `"stop"` means the model ended the answer on its own (or hit a stop sequence you set). `"length"` means it was cut off by a token limit, so the answer is incomplete.

## 5. Why caching matters (first evidence)
Time one embedding against one full answer from the LLM, on the same question.

In [10]:
import time

question = "How do I reset my password?"

model.encode(question, normalize_embeddings=True)  # warm-up, not timed

start = time.perf_counter()
for _ in range(20):
    model.encode(question, normalize_embeddings=True)
embed_ms = (time.perf_counter() - start) / 20 * 1000

# Ollama already loaded the model in section 4, so model loading is not counted.
start = time.perf_counter()
for _ in range(3):
    httpx.post(URL, json=request_body, timeout=120).raise_for_status()
llm_ms = (time.perf_counter() - start) / 3 * 1000

print(f"embedding: {embed_ms:.1f} ms (average of 20)")
print(f"LLM call:  {llm_ms:.1f} ms (average of 3)")
print(f"ratio:     {llm_ms / embed_ms:.0f}x")

embedding: 11.0 ms (average of 20)
LLM call:  3762.1 ms (average of 3)
ratio:     342x


This ratio is the core reason the project exists: a cache hit costs roughly the embedding plus the lookup, instead of a full generation. The lookup is not timed here, and these numbers are from a Mac running Ollama, not the vLLM GPU benchmark.

## 6. My explain-back

**What is an embedding, in one sentence I could say to an interviewer?**

An embedding turns a piece of text into a list of numbers (384 for bge-small) so that texts with similar meaning end up close together, which lets me compare meaning with simple math instead of comparing exact words.

**Why do paraphrases usually score below 0.98, and what does that mean for our threshold?**

Paraphrases use different words and sentence structure, so their vectors point in similar but not identical directions. In my notebook real paraphrases scored only 0.83–0.92. So a 0.98 threshold is very safe but would almost never give a cache hit. And a wrong pair (reset password vs reset username) scored 0.863, higher than one real paraphrase (0.834), so no single cut-off can separate right from wrong. That's why I tune the threshold on labelled data and use a verifier for the grey zone.

**What does finish_reason tell us, and why does the gateway only store "stop" answers?**

finish_reason tells why the model stopped writing. "stop" means it finished the answer on its own, so the answer is complete. "length" means it hit the token limit and the answer was cut off. The gateway only stores "stop" answers because a cut-off answer would be served incomplete to every future user who asks something similar.

**From section 5: how many times faster is the embedding than the LLM call on my Mac?**

About 368 times: around 10 ms to embed a question vs around 3,750 ms for the LLM to generate an answer. So checking the cache adds well under 1% to a miss, while a hit avoids the full generation. This isn't the full cache-hit speed-up, though, because the lookup and gateway time aren't included. The benchmark measures that.